# Block-level storage warming analysis

Consumes the per-block savings parquet emitted by
`access_savings_parser.py --per-block-out per_block_savings.parquet`.

Pipeline:
1. `python3 extract_block.py --start N --end M --out data` (writes `block_*.parquet` and `meta_*.parquet`)
2. `python3 access_savings_parser.py --dir data --windows 0,1,2,4,8,16,32,64,128,256,512,1024 --per-block-out data/per_block_savings.parquet`
3. Run this notebook (default reads from `./data/`; override via `BALWARMING_DIR=...`).

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = os.environ.get('BALWARMING_DIR', 'data')
PER_BLOCK = os.path.join(DATA_DIR, 'per_block_savings.parquet')

color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

def pick_windows(windows_present, candidates):
    """Return the candidates that are actually present in the dataset, preserving order."""
    s = set(windows_present)
    return [w for w in candidates if w in s]

In [ ]:
df = pd.read_parquet(PER_BLOCK).sort_values('block_number').reset_index(drop=True)
df['date'] = pd.to_datetime(df['timestamp'], unit='s')
windows_present = sorted(int(c[len('saving_W'):]) for c in df.columns if c.startswith('saving_W'))
print(f'{len(df)} blocks, range {df.block_number.min()}..{df.block_number.max()}, '
      f'{df.date.min()} -> {df.date.max()}; windows: {windows_present}')
df.head()

## Most-accessed (address, slot) pairs
Top-50 keys by access count across the loaded range. Read directly from the row-level parquets.

In [ ]:
block_files = sorted(glob.glob(os.path.join(DATA_DIR, 'block_*.parquet')),
                     key=lambda p: int(re.search(r'block_(\d+)', p).group(1)))
print(f'loading {len(block_files)} block parquets...')
rows = pd.concat([pd.read_parquet(f) for f in block_files], ignore_index=True)
n_blocks = rows.block_number.nunique()
n_txs = rows.groupby('block_number')['tx_index'].nunique().sum()
print(f'{len(rows):,} access rows from {n_blocks} blocks, {n_txs:,} txs')
rows['key'] = rows['address'].fillna('') + '|' + rows['storage_key'].fillna('')
pop = (rows.groupby('key')
           .agg(accesses_per_block=('block_number', lambda s: s.nunique() / n_blocks),
                accesses_per_tx=('key', 'count'))
           .assign(accesses_per_tx=lambda d: d['accesses_per_tx'] / n_txs)
           .sort_values('accesses_per_tx', ascending=False)
           .head(50)
           .reset_index())
pop[['address', 'storage_key']] = pop['key'].str.split('|', n=1, expand=True)
pop['Abbreviated Address'] = pop['address'].apply(lambda x: x[:6] + '...' + x[-4:] if x else '')
pop['Storage Key'] = pop['storage_key'].apply(lambda x: (x[:6] + '...' + x[-4:]) if x and len(x) > 12 else (x or ''))
pop = pop.rename(columns={'accesses_per_block': 'Accesses per Block', 'accesses_per_tx': 'Accesses per Transaction'})
pop[['Abbreviated Address', 'Storage Key', 'Accesses per Block', 'Accesses per Transaction']].round(4).head(20)

## Gas-used time series with hypothetical warming
Each line shows what block-level gas usage would look like under a given warming-window size. Window sizes are picked from `windows_present`.

In [ ]:
# Bin to hour if span is large; otherwise show per-block.
span_hours = (df.date.max() - df.date.min()).total_seconds() / 3600
if span_hours >= 4:
    df_t = df.copy()
    df_t['date'] = df_t['date'].dt.ceil('h')
    agg = df_t.groupby('date').agg(
        gas_used=('gas_used', 'mean'),
        block_count=('block_number', 'count'),
        **{f'saving_W{w}': (f'saving_W{w}', 'mean') for w in windows_present},
        max_saving=('max_saving', 'mean'),
    ).reset_index()
    x_axis_title = 'Date (hourly bin)'
else:
    agg = df.rename(columns={'block_number': 'date'}).copy()
    x_axis_title = 'Block number'

show_windows = pick_windows(windows_present, [0, 4, 16, 64, 256, 1024])
fig = go.Figure()
fig.add_trace(go.Scatter(x=agg['date'], y=agg['gas_used'], mode='lines',
                          name='Gas used (currently)', line=dict(width=2, color=color_palette[0])))
for i, w in enumerate(show_windows):
    fig.add_trace(go.Scatter(x=agg['date'], y=agg['gas_used'] - agg[f'saving_W{w}'],
                              mode='lines', name=f'W={w}',
                              line=dict(width=2, color=color_palette[(i+1) % len(color_palette)])))
fig.update_layout(title='Gas used vs hypothetical warming (multi-block windows)',
                  xaxis_title=x_axis_title, yaxis_title='Gas',
                  template='plotly_white', width=900, height=480, hovermode='x unified',
                  legend=dict(orientation='h', y=-0.2))
fig.show()

## Savings as % of block gas-used
Stacked area: intra-block warming (W=0) vs the largest multi-block window available.

In [ ]:
w_short = 0 if 0 in windows_present else min(windows_present)
w_long = max(w for w in windows_present if w >= 16) if any(w >= 16 for w in windows_present) else max(windows_present)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=-df[f'pct_saving_W{w_short}'], mode='lines',
                          name=f'Block-level (W={w_short})',
                          fill='tozeroy', line=dict(width=0),
                          fillcolor=hex_to_rgba(color_palette[0], 0.7)))
fig.add_trace(go.Scatter(x=df['date'], y=-df[f'pct_saving_W{w_long}'], mode='lines',
                          name=f'Multi-block (W={w_long})',
                          fill='tonexty', line=dict(width=0),
                          fillcolor=hex_to_rgba(color_palette[1], 0.7)))
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=0, xref='paper',
              line=dict(color='rgba(128,128,128,0.7)', dash='dash'))
fig.update_layout(title='Savings as % of block gas-used (negative = saved)',
                  xaxis_title='Date', yaxis_title='%',
                  template='plotly_white', width=900, height=480, hovermode='x unified',
                  legend=dict(orientation='h', y=-0.2))
fig.show()

## Saving potential vs window size (headline plot)
Median and mean of per-block percentage savings, across all loaded blocks, as a function of the warming-window size W. Log-x makes the saturation curve readable across many orders of magnitude.

In [ ]:
ws = sorted(w for w in windows_present if w >= 1)
med = [df[f'pct_saving_W{w}'].median() for w in ws]
avg = [df[f'pct_saving_W{w}'].mean() for w in ws]
max_med = df['pct_max_saving'].median()
max_mean = df['pct_max_saving'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=ws, y=med, mode='lines+markers', name='Median',
                          line=dict(color=color_palette[0]), marker=dict(size=8)))
fig.add_trace(go.Scatter(x=ws, y=avg, mode='lines+markers', name='Mean',
                          line=dict(color=color_palette[1]), marker=dict(size=8)))
fig.add_shape(type='line', x0=ws[0], x1=ws[-1], y0=max_med, y1=max_med,
              line=dict(color='rgba(128,128,128,0.7)', dash='dash'))
fig.add_annotation(x=ws[len(ws)//2], y=max_med,
                    text=f'Asymptote (median {max_med:.1f}%, mean {max_mean:.1f}%)',
                    showarrow=False, yshift=14, bgcolor='rgba(255,255,255,0.85)')
fig.update_layout(title='Multi-block warming potential',
                  xaxis=dict(title='Window size (blocks, log)', type='log'),
                  yaxis=dict(title='% savings of block gas used'),
                  template='plotly_white', width=900, height=520,
                  legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top'))
fig.show()
pd.DataFrame({'window': ws, 'median_pct': med, 'mean_pct': avg}).round(3)

## Per-opcode breakdown (largest window)

In [ ]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('access_savings_parser.py')))
from access_savings_parser import compute_block_gaps, per_opcode_breakdown
rows_g = compute_block_gaps(rows)
Wmax = max(windows_present)
per_opcode_breakdown(rows_g, Wmax).round(4)